# 01. Gaussian Processes: A Tutorial Introduction

**Why this notebook exists.** The rest of this series uses a Gaussian Process (GP) as a cheap stand-in for an expensive HUXt simulation (notebook 04). Before applying GPs to CMEs, it helps to see what a GP *is* on small, familiar problems. This notebook builds that intuition with two standard, non-CME examples; it complements the longer write-up in `runs/gp_surrogate/GP_TUTORIAL.md`.

**The one idea to take away.** A GP does not just draw a best-fit curve - it returns, at every point, a **prediction *and* an honest uncertainty**. The uncertainty is small near data you have seen and large where you have not. That "I know how much I don't know" is exactly what makes a GP useful for deciding where to spend the next expensive simulation (active learning, notebook 04, Task 4e).

**Roadmap:**

1. What a GP is, in words and in one equation.
2. The regression formula that produces the mean and uncertainty.
3. **Example 1** - the GP prior vs posterior on 1-D data: sample functions from each and watch the uncertainty collapse at the observations.
4. **Example 2** - a GP as a 2-input "simulator" emulator (the pattern notebook 04 uses).
5. A practical checklist for building a surrogate.

Set up a local environment first (see `setup.sh` / `README.md`):

```bash
python -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
python -m ipykernel install --user --name conecast --display-name "Python (conecast)"
```

In [ ]:
# --- Google Colab bootstrap (optional; does nothing when running locally) ---
import sys, os

if "google.colab" in sys.modules:
    REPO_URL = os.environ.get("CONECAST_REPO", "https://github.com/georgemilosh/conecast")
    REPO_DIR = "conecast"
    if not os.path.isdir(REPO_DIR):
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
    # HUXt and WSA+ are not preinstalled in Colab; the rest of the stack mostly is.
    os.system("pip install -q "
              "'huxt @ git+https://github.com/University-of-Reading-Space-Science/HUXt' "
              "wsaplus sunpy")
    # The WSA+ checkpoint (~317 MB) is fetched from Zenodo on demand by notebook 02
    # (scripts/fetch_wsaplus_checkpoint.py); notebook 01 needs no extra data.
    print("Colab bootstrap complete; cwd =", os.getcwd())
else:
    print("Not in Colab - using the local checkout.")

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
if (cwd / "scripts").exists():
    BASE_DIR = cwd
elif (cwd.parent / "scripts").exists():
    BASE_DIR = cwd.parent
else:
    # Fallback: assume the notebook is run from inside the repo.
    BASE_DIR = cwd

SCRIPT_DIR = BASE_DIR / "scripts"
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

print("BASE_DIR =", BASE_DIR)

## What Is A GP?

A Gaussian Process is a probability distribution over **functions**. Where an ordinary fit gives you one curve, a GP gives you a whole *family* of plausible curves consistent with the data, and summarizes that family by a mean and a spread:

$$
f(x) \sim \mathcal{GP}(m(x), k(x, x'))
$$

- The **mean function** `m(x)` is the expected function value (often taken as 0 after centering the data).
- The **kernel** `k(x, x')` is the heart of the method: it says how *correlated* two function values are based on how close their inputs are. Nearby inputs -> highly correlated outputs (the function is smooth); distant inputs -> nearly independent.

**Kernel intuition.** The kernel carries two knobs you will see throughout: an **output scale** (how far the function swings up and down) and a **length scale** (how far you must move in `x` before the function changes appreciably). A short length scale means a wiggly function; a long one means a slowly-varying function. In the CME work, fitting a separate length scale per parameter is what tells us which parameters the output is sensitive to.

After observing data, a GP gives:

- a posterior **mean** prediction,
- a posterior **uncertainty** (standard deviation),
- **wider** uncertainty far from training data (extrapolation),
- **narrower** uncertainty near informative data (interpolation).

> **Mental model.** Pin a flexible sheet to your data points. Between nearby pins the sheet is well-constrained (low uncertainty); far from any pin it flaps freely (high uncertainty). The kernel sets how stiff the sheet is.

## Regression Formula

For noisy observations

$$
y_i = f(x_i) + \epsilon_i, \quad \epsilon_i \sim \mathcal{N}(0, \sigma_n^2),
$$

the predictive distribution at a new point is Gaussian:

$$
f(x_*) \mid X, y, x_* \sim \mathcal{N}(\mu_*, \sigma_*^2).
$$

With covariance matrix `K` and covariance vector `k_*`,

$$
\mu_* = k_*^T (K + \sigma_n^2 I)^{-1} y
$$

and

$$
\sigma_*^2 = k(x_*, x_*) - k_*^T (K + \sigma_n^2 I)^{-1} k_*.
$$

**Reading the formulas in words:**

- `K` is the kernel evaluated between every pair of *training* points; `k_*` is the kernel between the new point and each training point. Both come straight from your kernel choice.
- The **mean** $\mu_*$ is a weighted average of the observed `y`, with more weight on training points whose inputs are close to $x_*$ (large `k_*`). Far from all data, `k_*` -> 0 and the mean falls back to the prior mean.
- The **variance** $\sigma_*^2$ starts at the prior variance `k(x_*, x_*)` and is *reduced* by whatever the data already explain. Near data, the subtracted term is large -> small uncertainty; far away it vanishes -> uncertainty returns to the prior.
- $\sigma_n^2$ is the assumed observation **noise**; adding it on the diagonal is what lets the mean curve pass *near* (not exactly through) noisy points.

> **Side note (cost).** That matrix inverse is $O(n^3)$ in the number of training points - fine for the hundreds of HUXt runs here, but the reason GPs are reserved for *expensive* models with modest sample counts rather than big data.

## Example 1: Prior And Posterior On 1-D Data

This is the canonical picture of a GP (compare Rasmussen & Williams, Fig. 2.2). A GP is first a **prior over functions** - before seeing any data it already says "the function is smooth, with this amplitude and length scale". We can *sample* random functions from that prior. **Conditioning** on a few observations turns the prior into a **posterior**: the subset of those functions that still pass through (or near) the data. We sample from the posterior too.

We use **noise-free** observations here on purpose. With no observation noise the posterior is forced *exactly* through each data point, so the uncertainty there collapses to zero - the lesson that was hidden when we added a noise term.

**Left panel (prior):** five functions drawn from `GP(0, k)` with the mean (flat 0) and the +/-2σ band. Every sample is a different but equally-plausible smooth curve, and the uncertainty is the **same everywhere** - we have no data yet.

**Right panel (posterior):** the same GP after conditioning on five red points. Now every sampled function threads the data, the mean is our best estimate, and the band **pinches to zero at each observation** and balloons between and beyond them.

> **What to look for:** the band touching zero at every red dot, and the posterior samples fanning out only where there is no data. (To model *noisy* data instead, add a `WhiteKernel` or set `alpha > 0`; the band then stays finite at the points - this is what notebook 04 does.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF

# A smooth ground truth we reveal at only a few points.
def truth(x):
    return np.sin(x) + 0.3 * x

x = np.linspace(0.0, 10.0, 400).reshape(-1, 1)

# Output-scale * squared-exponential kernel. optimizer=None fixes the hyperparameters
# (deterministic demo); alpha ~ 0 means noise-free, so the posterior interpolates exactly.
kernel = ConstantKernel(1.0) * RBF(length_scale=1.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, optimizer=None)

# --- PRIOR: sample functions before seeing any data ---
prior_samples = gp.sample_y(x, n_samples=5, random_state=1)
prior_mean, prior_std = gp.predict(x, return_std=True)

# --- POSTERIOR: condition on five noise-free observations ---
x_train = np.array([1.0, 3.0, 5.5, 7.0, 9.0]).reshape(-1, 1)
y_train = truth(x_train).ravel()
gp.fit(x_train, y_train)
post_samples = gp.sample_y(x, n_samples=5, random_state=1)
post_mean, post_std = gp.predict(x, return_std=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True, constrained_layout=True)

axes[0].fill_between(x.ravel(), prior_mean - 2 * prior_std, prior_mean + 2 * prior_std,
                     alpha=0.2, color="tab:blue", label="+/-2 std (95%)")
axes[0].plot(x, prior_samples, lw=1.0, alpha=0.8)
axes[0].plot(x, prior_mean, "k", lw=2, label="mean")
axes[0].set_title("Prior: functions before seeing data")
axes[0].set_xlabel("x"); axes[0].set_ylabel("f(x)"); axes[0].legend(loc="upper left")

axes[1].fill_between(x.ravel(), post_mean - 2 * post_std, post_mean + 2 * post_std,
                     alpha=0.2, color="tab:blue", label="+/-2 std (95%)")
axes[1].plot(x, post_samples, lw=1.0, alpha=0.8)
axes[1].plot(x, post_mean, "k", lw=2, label="mean")
axes[1].scatter(x_train, y_train, c="red", zorder=5, label="observations")
axes[1].set_title("Posterior: functions after conditioning on data")
axes[1].set_xlabel("x"); axes[1].legend(loc="upper left")

## Example 2: Two-Input Simulation

This is the pattern the CME work uses, in miniature. We pretend a synthetic function is an **expensive simulator** with two inputs - temperature and pressure - and build a GP **surrogate** for it:

1. **Space-filling design** (`LatinHypercube`): choose 45 input combinations that cover the 2-D box evenly (exactly what notebook 04's `design` step does in 5-D).
2. **Run the "simulator"** at those points and **scale** inputs/outputs.
3. **Fit** a 2-D GP (one length scale per input).
4. **Predict** on a dense grid and plot **two** surfaces: the mean response and the GP's uncertainty.

> **What to look for:** in the right-hand uncertainty panel, the valleys sit **on the design points** (white markers) and the ridges sit in the gaps between them. That map of "where am I unsure" is precisely what drives next-run selection in notebook 04 (Task 4e) - you add simulations where the surrogate is least certain.

In [ ]:
from scipy.stats import qmc
from sklearn.gaussian_process.kernels import Matern, WhiteKernel
from sklearn.preprocessing import StandardScaler

low = np.array([300.0, 1.0])
high = np.array([1200.0, 5.0])
sampler = qmc.LatinHypercube(d=2, seed=7)
X_design = qmc.scale(sampler.random(n=45), low, high)

def simulator(x):
    temperature, pressure = x
    return np.sin(temperature / 145.0) + 0.35 * pressure - 0.0000022 * (temperature - 820.0) ** 2

y_design = np.array([simulator(x) for x in X_design])

x_scaler = StandardScaler()
y_scaler = StandardScaler()
Xs = x_scaler.fit_transform(X_design)
ys = y_scaler.fit_transform(y_design.reshape(-1, 1)).ravel()

kernel = ConstantKernel(1.0) * Matern(length_scale=np.ones(2), nu=2.5) + WhiteKernel(noise_level=0.002)
gp2 = GaussianProcessRegressor(kernel=kernel, optimizer=None, random_state=8)
gp2.fit(Xs, ys)

temp = np.linspace(low[0], high[0], 90)
pressure = np.linspace(low[1], high[1], 80)
tt, pp = np.meshgrid(temp, pressure)
X_grid = np.column_stack([tt.ravel(), pp.ravel()])
mean_s, std_s = gp2.predict(x_scaler.transform(X_grid), return_std=True)
mean = y_scaler.inverse_transform(mean_s.reshape(-1, 1)).reshape(tt.shape)
std = (std_s * y_scaler.scale_[0]).reshape(tt.shape)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), constrained_layout=True)
im0 = axes[0].contourf(tt, pp, mean, levels=18)
axes[0].scatter(X_design[:, 0], X_design[:, 1], s=16, color="white", edgecolor="black")
axes[0].set_title("GP mean")
axes[0].set_xlabel("temperature [K]")
axes[0].set_ylabel("pressure [bar]")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].contourf(tt, pp, std, levels=18)
axes[1].scatter(X_design[:, 0], X_design[:, 1], s=16, color="white", edgecolor="black")
axes[1].set_title("GP uncertainty")
axes[1].set_xlabel("temperature [K]")
axes[1].set_ylabel("pressure [bar]")
fig.colorbar(im1, ax=axes[1])

## Practical Checklist

This is the recipe the rest of the series follows. Map each step to its notebook-04 task as you go:

1. Define input parameters and output quantities. *(the 5 Cone-CME parameters; hit + arrival time)*
2. Choose physically meaningful bounds. *(Task 1: priors + spans)*
3. Create a space-filling design. *(Task 1: Latin hypercube)*
4. Run the expensive model or experiment. *(Task 2: HUXt + detector)*
5. Scale inputs and outputs. *(Task 3)*
6. Fit the GP. *(Task 3: classifier + regressor)*
7. Validate with held-out data and uncertainty coverage. *(Task 3: holdout MAE)*
8. Use the GP for sensitivity analysis, uncertainty propagation, or active learning. *(Task 4a-4e)*

> **Common pitfalls.** Forgetting to scale (length scales become meaningless); too few design points for the dimensionality (the surrogate extrapolates wildly - watch the uncertainty); and trusting the mean where the GP reports large uncertainty. The uncertainty band is not decoration - it is the model telling you when to stop trusting it.

Useful references:

- Rasmussen and Williams, [Gaussian Processes for Machine Learning](https://gaussianprocess.org/gpml/chapters/)
- scikit-learn, [Gaussian Processes](https://scikit-learn.org/stable/modules/gaussian_process.html)
- Duvenaud, [The Kernel Cookbook](https://www.cs.toronto.edu/~duvenaud/cookbook/index.html)